# Bag-of-n-Grams

_Feature Engineering_

**Count word-pairs and word-triples, not just single words — so "not good" can become its own feature.**

Bag-of-words turns a document into a vector of word counts and forgets the order. "dog bites man" and "man bites dog" get the exact same vector. That lost order sometimes carries the whole meaning — especially negation: "good" and "not good" are opposites, but bag-of-words sees the same word "good" in both.

       Bag-of-n-grams recovers a little of that order. Instead of counting only single words, it also counts contiguous runs of $n$ words. Now "not good" is its own feature, separate from "good" and "not". A sentiment model can learn that the "not good" column predicts a bad review.

---

This notebook builds bag-of-n-grams up one step at a time: first we watch unigrams fail on negation, then we add bigrams and watch the same model recover. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

### Step 1 — Build a tiny labelled negation corpus

Every example here hinges on **negation**: the same adjective flips meaning when "not" sits in front of it. The plain positives/negatives teach the model that "good"/"bad" carry sentiment, and the "not good"/"not bad" rows teach that "not" flips it.

The held-out test set is *unseen sentences* that still contain these negation phrases, so neither model can win by memorising the training strings — it has to actually represent the phrase.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity

# A small REAL inline sentiment corpus (no downloads).
# label 1 = positive, 0 = negative.
train_texts = [
    # plain positives (label 1)
    "good", "this is good", "good film", "really good", "so good",
    # plain negatives (label 0)
    "bad", "this is bad", "bad film", "really bad", "so bad",
    # "not good" -> negative (label 0)
    "not good", "it was not good", "not good at all", "the acting is not good",
    # "not bad" -> positive (label 1)
    "not bad", "it was not bad", "not bad at all", "the acting is not bad",
]
train_labels = np.array(
    [1] * 5 +    # good ...
    [0] * 5 +    # bad ...
    [0] * 4 +    # not good ...
    [1] * 4      # not bad ...
)

# Held-out NEGATION sentences the model never saw verbatim.
test_texts = [
    "the movie was not good",      # neg
    "honestly not good",           # neg
    "a not good experience",       # neg
    "the movie was not bad",       # pos
    "honestly not bad",            # pos
    "a not bad experience",        # pos
]
test_labels = np.array([0, 0, 0, 1, 1, 1])  # not good = neg, not bad = pos

print("train examples:", len(train_texts), " test examples:", len(test_texts))

### Step 2 — Vectorise with unigrams and reproduce the problem

Plain bag-of-words (`ngram_range=(1, 1)`) counts only single words. So "good" and "not good" share the "good" column and differ by just the "not" column — their vectors look almost the same. We measure that with **cosine similarity** (high → easy to confuse), then train the classifier on these unigram features and score it on the held-out negation sentences. The accuracy is poor because the model has no way to see "not good" as a unit.

In [ ]:
# Vectorize with UNIGRAMS only (plain bag-of-words).
uni_vec = CountVectorizer(ngram_range=(1, 1))
uni_vec.fit(train_texts + test_texts)

# Raw vectors for "good" vs "not good": they share the "good" column.
v_good_u = uni_vec.transform(["good"]).toarray()[0]
v_notgood_u = uni_vec.transform(["not good"]).toarray()[0]

# Plot the two raw vectors over the columns either phrase uses.
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
words = uni_vec.get_feature_names_out()
keep = np.where((v_good_u + v_notgood_u) > 0)[0]  # only columns either phrase uses
x = np.arange(len(keep))
ax[0].bar(x - 0.2, v_good_u[keep], 0.4, label='"good"', color="#27ae60")
ax[0].bar(x + 0.2, v_notgood_u[keep], 0.4, label='"not good"', color="#c0392b")
ax[0].set_xticks(x)
ax[0].set_xticklabels(words[keep], rotation=0)
ax[0].set_ylabel("count")
ax[0].set_title("STEP 2: UNIGRAM bag-of-words\n'good' vs 'not good' — only differ by one 'not' column")
ax[0].legend()

# How similar do "good" and "not good" look? Cosine of their unigram vectors.
cos_uni = cosine_similarity(v_good_u.reshape(1, -1), v_notgood_u.reshape(1, -1))[0, 0]
print(f'STEP 3  UNIGRAM cosine("good", "not good") = {cos_uni:.3f}  (high -> looks similar)')

# Train the SAME classifier on unigram features, score held-out negation examples.
X_train_u = uni_vec.transform(train_texts)
clf_u = LogisticRegression(max_iter=1000, random_state=0).fit(X_train_u, train_labels)
uni_acc = clf_u.score(uni_vec.transform(test_texts), test_labels)
print(f"STEP 3 PROBLEM  unigram bag-of-words, held-out negation accuracy = {uni_acc:.3f}")

### Step 3 — Add bigrams and show the fix

Now switch to `ngram_range=(1, 2)`: keep the unigrams **and** add adjacent word pairs. "not good" gets its own column, so it no longer shares its non-zero features with "good" — the cosine similarity drops. We refit the *same* `LogisticRegression`, only the features changed, and the held-out negation accuracy jumps because the model can finally weight "not good" as negative. The right-hand plot of the figure shows the new feature columns, and the printed line summarises the before/after.

In [ ]:
# ngram_range=(1, 2): keep unigrams AND adjacent pairs.
bi_vec = CountVectorizer(ngram_range=(1, 2))
bi_vec.fit(train_texts + test_texts)
v_good_b = bi_vec.transform(["good"]).toarray()[0]
v_notgood_b = bi_vec.transform(["not good"]).toarray()[0]

# Plot the engineered vectors on the right axis of the same figure.
feats = bi_vec.get_feature_names_out()
keep_b = np.where((v_good_b + v_notgood_b) > 0)[0]
xb = np.arange(len(keep_b))
ax[1].bar(xb - 0.2, v_good_b[keep_b], 0.4, label='"good"', color="#27ae60")
ax[1].bar(xb + 0.2, v_notgood_b[keep_b], 0.4, label='"not good"', color="#c0392b")
ax[1].set_xticks(xb)
ax[1].set_xticklabels(feats[keep_b], rotation=20, ha="right")
ax[1].set_ylabel("count")

# Cosine of the bigram vectors: "not good" is now its own feature.
cos_bi = cosine_similarity(v_good_b.reshape(1, -1), v_notgood_b.reshape(1, -1))[0, 0]
ax[1].set_title(f"STEP 4: with BIGRAMS (1,2)\n'not good' is its own feature — cosine drops to {cos_bi:.2f}")
ax[1].legend()
plt.tight_layout()
plt.show()

print(f'STEP 4  BIGRAM  cosine("good", "not good") = {cos_bi:.3f}  (lower -> now distinct)')

# SAME LogisticRegression, now on (1,2)-gram features.
X_train_b = bi_vec.transform(train_texts)
clf_b = LogisticRegression(max_iter=1000, random_state=0).fit(X_train_b, train_labels)
bi_acc = clf_b.score(bi_vec.transform(test_texts), test_labels)
print(f"STEP 5 FIX      bag-of-bigrams, held-out negation accuracy = {bi_acc:.3f}")

print(f"PROBLEM (unigram): {uni_acc:.3f}   ->   FIX (bigram): {bi_acc:.3f}")

## Reference implementation — scikit-learn (CountVectorizer), pandas

### Step 1 — Load Yelp reviews and count n-grams of each size

This is the real Yelp review corpus from the Feature Engineering book. We load the raw review strings, then fit a `CountVectorizer` at three separate widths — unigrams only, bigrams only, trigrams only — and print how many distinct features each produces. Watch the feature count explode: bigrams give roughly 12x more columns than unigrams, trigrams roughly 56x.

In [ ]:
# pip install scikit-learn pandas
# Dataset: Yelp Dataset Challenge (round 6) reviews, yelp.com/dataset
# Book code: github.com/alicezheng/feature-engineering-book (Chapter 3)
import json
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# --- load the Yelp review text (one JSON object per line) ---
review_file = "yelp_academic_dataset_review.json"
reviews = []
with open(review_file) as f:
    for line in f:
        reviews.append(json.loads(line))
review_df = pd.DataFrame(reviews)
texts = review_df["text"]          # the raw review strings

# --- Bag-of-words: single words only (unigrams) ---
bow = CountVectorizer()            # default ngram_range=(1, 1)
bow.fit(texts)
print("unigram features :", len(bow.get_feature_names_out()))   # ~ 29,000

# --- Bag-of-bigrams: word-pairs only ---
bigram = CountVectorizer(ngram_range=(2, 2))
bigram.fit(texts)
print("bigram features  :", len(bigram.get_feature_names_out())) # ~ 357,000

# --- Bag-of-trigrams: word-triples only ---
trigram = CountVectorizer(ngram_range=(3, 3))
trigram.fit(texts)
print("trigram features :", len(trigram.get_feature_names_out())) # ~ 1,627,000

### Step 2 — The useful setting: unigrams and bigrams together

In practice you keep both single words and pairs at once with `ngram_range=(1, 2)`. This is the setting that recovers negation: "not good" now has its own vocabulary index, distinct from "good". We confirm both indices exist, then look at the cost — a huge, mostly-zero document-by-n-gram matrix.

In [ ]:
# --- The useful setting: unigrams AND bigrams together ---
uni_bi = CountVectorizer(ngram_range=(1, 2))
X = uni_bi.fit_transform(texts)    # sparse document x n-gram count matrix
print("uni+bigram feats :", len(uni_bi.get_feature_names_out()))

# negation now has its own feature: "not good" is distinct from "good"
vocab = uni_bi.vocabulary_
print('"good" index     :', vocab.get("good"))
print('"not good" index :', vocab.get("not good"))

# the cost is sparsity: a huge, mostly-zero matrix
print("matrix shape     :", X.shape, " stored nonzeros:", X.nnz)

### Step 3 — Tame the explosion by pruning

Most of those millions of n-grams appear in only a handful of reviews — rare noise that bloats the matrix without helping. `min_df=10` drops any n-gram seen in fewer than 10 documents, and `max_features=50000` caps the vocabulary at the most frequent columns. The result is a manageable feature set that still keeps the common, useful phrases.

In [ ]:
# tame the explosion: keep only n-grams seen in many reviews
pruned = CountVectorizer(ngram_range=(1, 2), min_df=10, max_features=50000)
pruned.fit(texts)
print("pruned features  :", len(pruned.get_feature_names_out()))

## Visualize the data & results

_On a tiny real corpus of reviews (one with the negation "not good"), how fast does the number of distinct features grow as we widen ngram_range from (1,1) to (1,2) to (1,3)?_

### Step 1 — Count distinct features as the n-gram range widens

Here we use a tiny six-review corpus (one review carries the negation "not good") so the numbers are easy to read. We fit a `CountVectorizer` at three ranges — (1,1), (1,2), (1,3) — and record how many distinct features each yields. Even on six short reviews the count climbs steeply as we add wider n-grams.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# six short REAL reviews; one carries a negation ("not good")
corpus = [
    "the food is good",
    "the food is not good",
    "great service and great food",
    "service was not great",
    "i would not come back",
    "good food and good service",
]

# count distinct features for three n-gram ranges
counts = {}
for rng in [(1, 1), (1, 2), (1, 3)]:
    vec = CountVectorizer(ngram_range=rng)
    vec.fit(corpus)
    counts[rng] = len(vec.get_feature_names_out())
    print("ngram_range", rng, "->", counts[rng], "features")
# ngram_range (1, 1) -> 12 features
# ngram_range (1, 2) -> 31 features
# ngram_range (1, 3) -> 45 features
# (sklearn's default token pattern drops 1-char tokens like "i")

### Step 2 — Confirm negation is recovered, and measure the sparsity cost

With the (1,2) bag fitted, we check that both "good" and "not good" are in the vocabulary — the negation phrase survives as its own feature. Then we transform the corpus and compute the matrix sparsity: more columns with the same few hits per row means a wider, emptier matrix. That is the price you pay for the local-order signal.

In [ ]:
# the (1,2) bag now has "not good" as its OWN feature, distinct from "good"
vec12 = CountVectorizer(ngram_range=(1, 2)).fit(corpus)
vocab = vec12.vocabulary_
print('"good" in vocab     :', "good" in vocab)        # True
print('"not good" in vocab :', "not good" in vocab)    # True  <- recovered negation

# the cost: a wider, sparser document x feature matrix
X = vec12.transform(corpus)
sparsity = 100 * (1 - X.nnz / (X.shape[0] * X.shape[1]))
print("matrix shape", X.shape, "nonzeros", X.nnz, "sparsity %.1f%%" % sparsity)

### Step 3 — Plot the feature-count explosion

Finally we draw the three feature counts as a bar chart. The bars make the trade-off vivid: each step from (1,1) to (1,2) to (1,3) adds more distinct features, the cost of capturing longer local phrases.

In [ ]:
import matplotlib.pyplot as plt

ranges = ["(1,1)", "(1,2)", "(1,3)"]
vals = [counts[(1, 1)], counts[(1, 2)], counts[(1, 3)]]
plt.bar(ranges, vals, color=["#4ea1ff", "#7ee787", "#ffb454"])
plt.ylabel("number of distinct features")
plt.title("Feature count explodes as ngram_range widens")
for i, v in enumerate(vals):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The review i would not order this again has 6 words. How many bigrams ($n=2$) and trigrams ($n=3$) does it produce? Which bigram carries the negation?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use $L - n + 1$ with $L = 6$. Bigrams: $6 - 2 + 1 = 5$. Trigrams: $6 - 3 + 1 = 4$. — _Slide a window of width $n$ across the 6 words._
- List the bigrams: "i would", "would not", "not order", "order this", "this again". — _Each adjacent pair is one bigram feature._
- "not order" (and "would not") is the negation-bearing pair. — _Bag-of-words would split "not" from "order"; the bigram keeps the negation as one feature._

**Answer:** 5 bigrams and 4 trigrams. The bigram "not order" (helped by "would not") carries the negation that plain bag-of-words loses.

</details>

**Problem 2.** On Yelp the book reports about 29,000 unigrams, 357,000 bigrams, and 1,627,000 trigrams. Roughly how many times more features do bigrams and trigrams give versus unigrams, and what does that imply about the matrix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Bigrams vs unigrams: $357\text{k} / 29\text{k} \approx 12$. Trigrams vs unigrams: $1{,}627\text{k} / 29\text{k} \approx 56$. — _Divide each count by the unigram count to get the multiplier._
- So adding bigrams makes the feature space ~12x wider; trigrams ~56x wider. — _Distinct n-grams grow far faster than the vocabulary._
- Each document still contains only a handful of those n-grams, so almost every column is zero. — _More columns with the same few hits per row means a much sparser matrix._

**Answer:** Bigrams ~12x and trigrams ~56x more features than unigrams. The matrix becomes far wider and much sparser — the cost you pay for the extra local-order signal. Prune with min_df / max_features.

</details>

**Problem 3.** A sentiment model on bag-of-words keeps calling "the food was not good" positive. Why, and what single change to the vectorizer most directly fixes it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Bag-of-words ($n=1$) has separate "not" and "good" features; the strong positive weight on "good" wins. — _Unigrams cannot represent that "not" negates "good" — order is discarded._
- Switch to a bag-of-n-grams that includes bigrams: CountVectorizer(ngram_range=(1,2)). — _This adds the "not good" feature, which the model can weight negatively._
- Optionally add min_df to prune the new rare bigrams and control the blow-up. — _Most added bigrams are rare noise; pruning keeps the matrix manageable._

**Answer:** Unigrams split "not" from "good", so the positive "good" weight dominates. Set ngram_range=(1,2) so "not good" becomes its own feature the model can penalize — and use min_df to tame the feature explosion.

</details>